|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 0:</h2>|<h1>From a Program to a Model<h1>|
|<h2>Section:</h2>|<h1>The map<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: which layer?<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

# Which layer?

In your web services, the first question after a failure is "which layer?".
The answer tells you which file to open. Here is the same question for an
engine.

Each ticket below is a symptom, as a user or a dashboard reports it. For each
one, name the layer where you would look first. Use one of these five:

    ROUTER, SERVICE, REPO, DB DRIVER, DB ENGINE

You do not know how to fix any of them yet. You do not need to. A good first
guess of the layer is the skill. Use the map of the first notebook of this
section, or `docs/MAP.md`.

### Given

- The five layers of the map, and what each one does.
- Eight symptoms, in the words of the person who saw them.

### Asked

- For each symptom, the one layer where you look first.
- You know that you are correct when the check prints `8 of 8`.

In [ ]:
# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import hashlib

def check(answers):
    """The answers are stored as hashes, so that this cell does not give them away."""
    right = 0
    for ticket, layer in answers.items():
        digest = hashlib.sha256(f'{ticket}:{str(layer).strip().upper()}'.encode()).hexdigest()[:12]
        ok = digest == EXPECTED[ticket]
        right += ok
        print(f'  ticket {ticket}: {"correct" if ok else "not yet"}')
    print(f'{right} of {len(EXPECTED)}')

EXPECTED = {'a': '9a7ebd9f6719', 'b': 'cd9630e82670', 'c': '6e47695d606f', 'd': '980be7680de8', 'e': '1c8433df7ab7', 'f': 'b158c38f4e3a', 'g': 'ab44e4510e41', 'h': '1d613a55414d'}

### The tickets

**a.** When one user pastes a 20-page document, every other user sees their stream stop for a moment.

**b.** The answers read well, but sometimes a new answer continues the topic of an earlier user.

**c.** The model writes its private reasoning into the answer, but only through the chat endpoint.

**d.** At batch 1, the new kernel for **attention** (the part of the model that reads the memo table) uses only a small part of the memory **bandwidth** (bytes per second) of the card. Each thread of it is fast.

**e.** After a change to how the step is captured, one user's answer changes a few tokens after another user joins the batch.

**f.** All the streams stop together for about 20 ms at a time, and each stop matches one step of the engine.

**g.** With the same traffic, the server holds fewer requests every hour. No client disconnects.

**h.** Characters of Hindi text appear broken in the stream, but the final text is correct.

In [ ]:
ANSWERS = {
    'a': 'SERVICE',
    'b': 'REPO',
    'c': 'ROUTER',
    'd': 'DB ENGINE',
    'e': 'DB DRIVER',
    'f': 'SERVICE',
    'g': 'REPO',
    'h': 'SERVICE',
}
check(ANSWERS)

### Why

- **a. SERVICE.** The scheduler put the whole long prompt into one step, and every other request waits for that step. Stage 11 cuts a long prompt into chunks.
- **b. REPO.** A request got memo pages that belong to another request: a leak, or a wrong key in the prefix cache. Stage 09 has this bug on purpose.
- **c. ROUTER.** The router builds the chat prompt from the messages. A wrong chat template, or a missing option of it, changes what the model sees. Stage 27.
- **d. DB ENGINE.** At batch 1 the kernel has too little parallel work, and most of the GPU waits. Stage 08c splits the work so that the whole GPU has some.
- **e. DB DRIVER.** The runner fills the empty rows of a fixed-size batch. If an empty row writes into page 0, it overwrites the memo of a real request. Stage 23.
- **f. SERVICE.** The engine loop runs its step inside the event loop, so nothing else can run during the step. Stage 27 moves the step to its own thread.
- **g. REPO.** Pages that nobody uses never go back to the free list: a leak in the repo. The stage 22 checks count the free pages after each run.
- **h. SERVICE.** The detokenizer turns one token at a time into text, and one character can use more than one token. Stage 14.

### After the check

For each ticket that you got wrong, find the box on the map that you should
have chosen, and read what it does. Two of the tickets have the same layer as
another ticket, but a different cause. The layer tells you where to look. It
does not tell you what you will find.